In [0]:
jdbc_url = "jdbc:postgresql://ep-delicate-cake-axb41sh7.c-4.us-east-2.aws.neon.tech:5432/neondb?sslmode=require"

connection_properties = {
    "user": "neondb_owner",
    "password": "npg_jAgkVH0D5ndX",
    "driver": "org.postgresql.Driver"
}

agences_df = spark.read.jdbc(url=jdbc_url, table="agences", properties=connection_properties)
produits_df = spark.read.jdbc(url=jdbc_url, table="produits", properties=connection_properties)
canevas_df = spark.read.jdbc(url=jdbc_url, table="canevas", properties=connection_properties)

agences_df.printSchema()
canevas_df.show(5)

In [0]:
print("Agences:", agences_df.count())
print("Produits:", produits_df.count())
print("Canevas:", canevas_df.count())

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

df = (
    canevas_df
    .join(agences_df, on="code_agence", how="left")
    .join(produits_df, on="code_produit", how="left")
    .withColumn("annee", F.year("date_creation"))
    .withColumn("trimestre", F.concat(F.lit("T"), F.quarter("date_creation"), F.lit(" "), F.year("date_creation")))
    .withColumn("age", F.round(F.datediff("date_creation", "date_naissance") / 365.25, 1))
    .withColumn("delai_mep_jours", F.datediff("date_mep", "date_creation"))
    .withColumn(
        "taux_endettement",
        F.when(F.col("revenu_annuel") > 0, F.col("retenue_mensuelle") * 12 / F.col("revenu_annuel"))
    )
    .withColumn(
        "tranche_age",
        F.when(F.col("age") < 25, "< 25 ans")
         .when(F.col("age") < 35, "25-34 ans")
         .when(F.col("age") < 45, "35-44 ans")
         .when(F.col("age") < 55, "45-54 ans")
         .when(F.col("age") < 65, "55-64 ans")
         .otherwise("65 ans et +")
    )
)

nb_avant = df.count()


In [0]:
kpi_qualite_donnees = df.agg(
    F.count("*").alias("nb_dossiers_total"),
    F.sum(F.col("sexe").isNull().cast("int")).alias("sexe_manquant"),
    F.sum(((F.col("age") > 100) | (F.col("age") < 15)).cast("int")).alias("age_aberrant"),
    F.sum(((F.col("decision_finale") == "ACCORD") & F.col("date_mep").isNull()).cast("int")).alias("accord_sans_mep"),
    F.sum((F.col("delai_mep_jours") < 0).cast("int")).alias("delai_mep_negatif"),
    F.sum((F.col("montant_sollicite") >= 1_000_000_000).cast("int")).alias("montant_extreme"),
    F.sum((F.col("taux_endettement") > 3).cast("int")).alias("endettement_aberrant"),
)

# Nettoyage : on filtre les anomalies évidentes pour les tables KPI
df_clean = df.filter(
    (F.col("montant_sollicite") < 1_000_000_000) &
    (F.col("age").between(15, 100) | F.col("age").isNull()) &
    ((F.col("delai_mep_jours") >= 0) | F.col("delai_mep_jours").isNull())
)

nb_apres = df_clean.count()
print(f"Lignes avant nettoyage : {nb_avant:,}  |  après : {nb_apres:,}  |  écartées : {nb_avant - nb_apres:,}")


In [0]:
kpi_global = df_clean.agg(
    F.count("*").alias("nb_dossiers"),
    F.round(F.avg((F.col("decision_finale") == "ACCORD").cast("int")) * 100, 1).alias("taux_accord_pct"),
    F.round(F.avg((F.col("decision_finale") == "REJET").cast("int")) * 100, 1).alias("taux_rejet_pct"),
    F.sum("montant_sollicite").alias("montant_total"),
    F.round(F.avg("montant_sollicite"), 0).alias("montant_moyen"),
    F.round(F.avg("delai_mep_jours"), 1).alias("delai_moyen_mep_jours"),
    F.round(F.avg(F.when(F.col("taux_endettement") < 3, F.col("taux_endettement"))) * 100, 1).alias("taux_endettement_moyen_pct"),
    F.countDistinct("code_agence").alias("nb_agences_actives"),
    F.countDistinct("code_produit").alias("nb_produits_distincts"),
)

In [0]:
dim_agence = (
    agences_df
    .select("code_agence", "libelle_agence", "libelle_direction_regionale")
    .distinct()
)
display(dim_agence)

In [0]:
dim_produit = (
    produits_df
    .select("code_produit", "libelle_produit")
    .distinct()
)
display(dim_produit)


In [0]:
date_bounds = df_clean.agg(F.min("date_creation").alias("d_min"), F.max("date_creation").alias("d_max")).collect()[0]

dim_date = (
    spark.sql(f"SELECT explode(sequence(to_date('{date_bounds['d_min']}'), to_date('{date_bounds['d_max']}'), interval 1 day)) AS date_key")
    .withColumn("annee", F.year("date_key"))
    .withColumn("trimestre", F.concat(F.lit("T"), F.quarter("date_key")))
    .withColumn("mois", F.month("date_key"))
    .withColumn("nom_mois", F.date_format("date_key", "MMMM"))
)
display(dim_date)


In [0]:
dim_decision = (
    df_clean
    .select("centre_decision", "decision_finale")
    .distinct()
    .withColumn("id_decision", F.monotonically_increasing_id())
)
display(dim_decision)

In [0]:
dim_profil_emprunteur = (
    df_clean
    .select("sexe", "tranche_age", "profession")
    .distinct()
    .withColumn("id_profil", F.row_number().over(
        Window.orderBy("sexe", "tranche_age", "profession")
    ))
    .select("id_profil", "sexe", "tranche_age", "profession")
)
display(dim_profil_emprunteur)


In [0]:
fact_credit = (
    df_clean
    .join(dim_decision, on=["centre_decision", "decision_finale"], how="left")
    .join(dim_profil_emprunteur, on=["sexe", "tranche_age", "profession"], how="left")
    .select(
        "num_canevas",
        "code_agence",                              # clé -> dim_agence
        "code_produit",                              # clé -> dim_produit
        F.col("date_creation").alias("date_key"),    # clé -> dim_date
        "id_decision",                                # clé -> dim_decision
        "id_profil",                                  # clé -> dim_profil_emprunteur
        "montant_sollicite",
        "revenu_annuel",
        "retenue_mensuelle",
        "taux_endettement",
        "delai_mep_jours",
        F.when(F.col("decision_finale") == "ACCORD", 1).otherwise(0).alias("est_accord"),
        F.when(F.col("decision_finale") == "REJET", 1).otherwise(0).alias("est_rejet"),
    )
)
display(fact_credit)


In [0]:

star_schema = {
    "dim_agence": dim_agence,
    "dim_produit": dim_produit,
    "dim_date": dim_date,
    "dim_decision": dim_decision,
    "dim_profil_emprunteur": dim_profil_emprunteur,
    "fact_credit": fact_credit,
}

for name, frame in star_schema.items():
    frame.write.format("delta").mode("overwrite").saveAsTable(f"credit_analysis.{name}")
    print(f"  {name} : {frame.count():,} lignes")

print("Schéma en étoile créé dans credit_analysis.")

# COMMAND ----------
spark.sql("SHOW TABLES IN credit_analysis").show(truncate=False)

